In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

In [0]:
# =====================================================
# Sample Data
# =====================================================

players_data = [
    (15, 1), (25, 1), (30, 1), (45, 1),
    (10, 2), (35, 2), (50, 2),
    (20, 3), (40, 3)
]

matches_data = [
    (1, 15, 45, 3, 0),
    (2, 30, 25, 1, 2),
    (3, 30, 15, 2, 0),
    (4, 40, 20, 5, 2),
    (5, 35, 50, 1, 1)
]

players_df = spark.createDataFrame(
    players_data,
    ["player_id", "group_id"]
)

matches_df = spark.createDataFrame(
    matches_data,
    [
        "match_id",
        "first_player",
        "second_player",
        "first_score",
        "second_score"
    ]
)

players_df.show()
matches_df.show()

+---------+--------+
|player_id|group_id|
+---------+--------+
|       15|       1|
|       25|       1|
|       30|       1|
|       45|       1|
|       10|       2|
|       35|       2|
|       50|       2|
|       20|       3|
|       40|       3|
+---------+--------+

+--------+------------+-------------+-----------+------------+
|match_id|first_player|second_player|first_score|second_score|
+--------+------------+-------------+-----------+------------+
|       1|          15|           45|          3|           0|
|       2|          30|           25|          1|           2|
|       3|          30|           15|          2|           0|
|       4|          40|           20|          5|           2|
|       5|          35|           50|          1|           1|
+--------+------------+-------------+-----------+------------+



In [0]:
# =====================================================
# Create Player-Level Scores
# =====================================================

first_player_scores_df = (
    matches_df
    .select(
        F.col("first_player").alias("player_id"),
        F.col("first_score").alias("score")
    )
)

second_player_scores_df = (
    matches_df
    .select(
        F.col("second_player").alias("player_id"),
        F.col("second_score").alias("score")
    )
)

player_scores_df = (
    first_player_scores_df
    .unionByName(second_player_scores_df)
)

player_scores_df.show()

+---------+-----+
|player_id|score|
+---------+-----+
|       15|    3|
|       30|    1|
|       30|    2|
|       40|    5|
|       35|    1|
|       45|    0|
|       25|    2|
|       15|    0|
|       20|    2|
|       50|    1|
+---------+-----+



In [0]:
# =====================================================
# Calculate Total Score Per Player
# =====================================================

player_totals_df = (
    player_scores_df
    .groupBy("player_id")
    .agg(
        F.sum("score").alias("total_score")
    )
)

# Include players who may not have played any matches
player_totals_df = (
    players_df
    .join(player_totals_df, "player_id", "left")
    .fillna(0, subset=["total_score"])
)

player_totals_df.show()

+---------+--------+-----------+
|player_id|group_id|total_score|
+---------+--------+-----------+
|       15|       1|          3|
|       25|       1|          2|
|       30|       1|          3|
|       45|       1|          0|
|       10|       2|          0|
|       35|       2|          1|
|       50|       2|          1|
|       20|       3|          2|
|       40|       3|          5|
+---------+--------+-----------+



In [0]:
# =====================================================
# Find Group Winner
# Highest score wins
# If tied, lower player_id wins
# =====================================================

group_rank_window = (
    Window
    .partitionBy("group_id")
    .orderBy(
        F.col("total_score").desc(),
        F.col("player_id").asc()
    )
)

group_winners_df = (
    player_totals_df
    .withColumn(
        "rank",
        F.row_number().over(group_rank_window)
    )
    .filter(F.col("rank") == 1)
    .select("group_id", "player_id")
    .orderBy("group_id")
)

group_winners_df.show()

+--------+---------+
|group_id|player_id|
+--------+---------+
|       1|       15|
|       2|       35|
|       3|       40|
+--------+---------+

